# Stimulus/session dataframe sandbox

This notebook rebuilds the mouse-arena session dataframe step by step so the timing assumptions are visible and editable.

Current sandbox focus: compare behavior `sync_events.csv` `game_frame` timestamps to the ephys-recorded Open Ephys/NI-DAQ digital events loaded with `open_ephys.analysis.Session`.

Policy: ephys TTL timestamps are ground truth. Behavior timestamps are mapped onto the ephys timebase only after we validate the alignment method.

In [ ]:
from pathlib import Path
import json
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from open_ephys.analysis import Session

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)

SESSION_FOLDER = Path(r"C:\Users\jordan\Desktop\mouse_arena_recordings\jlh602026-06-29_14-31-15")
RECORDING_FOLDER = SESSION_FOLDER / "Record Node 101" / "experiment1" / "recording1"
BEHAVIOR_FOLDER = Path(r"\\denmanlab\s1\behavior\mouse_arena\data\jlh60\2026-06-29_14-29-26")
OUTPUT_FOLDER = RECORDING_FOLDER / "nwb"

EPHYS_STREAM_NAME = "PXI-6133"
TTL_LINES = {
    "trial_start": 5,
    "reward": 4,
    "game_frame": 6,
}

def OE(path):
    session = Session(path)
    return session.recordnodes[0].recordings[0]

print(SESSION_FOLDER)
print(RECORDING_FOLDER)
print(BEHAVIOR_FOLDER)
print(OUTPUT_FOLDER)

## 1. Load behavior files

`events.csv` is the compact behavior event table. `sync_events.csv` contains the game-generated sync pulses that should correspond to the NI-DAQ TTL pulses.

In [ ]:
behavior_events.head()

In [ ]:
behavior_events = pd.read_csv(BEHAVIOR_FOLDER / "events.csv")
sync_events = pd.read_csv(BEHAVIOR_FOLDER / "sync_events.csv")
session_meta = json.loads((BEHAVIOR_FOLDER / "session_meta.json").read_text())

print("behavior_events", behavior_events.shape)
display(behavior_events["event_type"].value_counts(dropna=False).rename("count"))
display(behavior_events.head())

print("sync_events", sync_events.shape)
display(sync_events.groupby(["event_type", "line_name", "line_index"]).size().rename("count").reset_index())
display(session_meta)

In [ ]:
sync_events.head()

## 2. Load ephys-recorded digital events with Open Ephys tools

Use `open_ephys.analysis.Session` to load the Open Ephys recording and inspect `recording.events`. We filter to the NI-DAQ stream (`PXI-6133`) and rising edges (`state == 1`).

The behavior app reports sync output line indices as `game_frame=2`, `trial_start=4`, and `reward=3`, while the Open Ephys events dataframe records the corresponding ephys/NI-DAQ digital lines observed in this recording as trial starts on line 5, rewards on line 4, and the game/frame candidate on line 6.

In [ ]:
digital_events.head()

In [ ]:
recording = OE(SESSION_FOLDER)
events_df = recording.events.copy()

print("events_df", events_df.shape)
display(events_df.head())
display(events_df.groupby(["stream_name", "line", "state"]).size().rename("count").reset_index())

# Get recording-relative time zero from the Open Ephys continuous stream.
nidaq_continuous = next(item for item in recording.continuous if EPHYS_STREAM_NAME in item.name)
time_zero = float(nidaq_continuous.timestamps[0])

ephys_events = events_df.loc[
    events_df["stream_name"].eq(EPHYS_STREAM_NAME) & events_df["state"].eq(1)
].copy()

rows = []
for event_name, line in TTL_LINES.items():
    hits = ephys_events.loc[ephys_events["line"].eq(line)].sort_values("timestamp")
    for ordinal, (_, event) in enumerate(hits.iterrows(), start=1):
        rows.append({
            "event_name": event_name,
            "line": int(line),
            "edge": "rising",
            "ordinal": ordinal,
            "timestamp": float(event["timestamp"]),
            "time": float(event["timestamp"] - time_zero),
            "sample_number": int(event["sample_number"]),
            "stream_name": event["stream_name"],
        })

digital_events = pd.DataFrame(rows)

print("time_zero", time_zero)
display(digital_events.groupby(["event_name", "line"]).size().rename("count").reset_index())
display(digital_events.head())

## 3. Fit behavior clock to ephys clock

The primary fit uses dense `game_frame` sync events only. Behavior `game_frame` timestamps are mapped onto the ephys timebase, nearest ephys-recorded game-frame TTLs are matched, and the frame-derived fit is refined from those matches.

Trial starts and rewards are deliberately held out as validation signals, not used to seed the clock fit.

Ephys TTL times remain ground truth. This fit maps behavior-only timestamps onto the ephys clock.

In [ ]:
behavior_game_frame = sync_events.loc[sync_events["event_type"].eq("game_frame"), "t"].to_numpy(dtype=float)
ephys_frame = digital_events.loc[digital_events["event_name"].eq("game_frame"), "time"].to_numpy(dtype=float)

def _nearest_unique_frame_matches(behavior_times, ephys_times, slope, intercept, max_abs_residual_s):
    """Map behavior frame times into ephys time and keep one nearest ephys edge per behavior frame."""
    mapped = slope * behavior_times + intercept
    right = np.searchsorted(ephys_times, mapped, side="left")

    rows = []
    for behavior_index, right_index in enumerate(right):
        candidates = []
        if 0 <= right_index < len(ephys_times):
            candidates.append(right_index)
        if 0 <= right_index - 1 < len(ephys_times):
            candidates.append(right_index - 1)
        if not candidates:
            continue
        ephys_index = min(candidates, key=lambda idx: abs(ephys_times[idx] - mapped[behavior_index]))
        residual = ephys_times[ephys_index] - mapped[behavior_index]
        if abs(residual) <= max_abs_residual_s:
            rows.append((behavior_index, ephys_index, behavior_times[behavior_index], ephys_times[ephys_index], mapped[behavior_index], residual))

    matches = pd.DataFrame(rows, columns=[
        "behavior_frame_index", "ephys_frame_index", "behavior_time", "ephys_time", "mapped_time", "residual_seconds"
    ])
    if matches.empty:
        return matches

    # If two behavior frames point to the same ephys edge, keep the smaller absolute residual.
    matches = (
        matches.assign(abs_residual_seconds=matches["residual_seconds"].abs())
        .sort_values("abs_residual_seconds")
        .drop_duplicates("ephys_frame_index", keep="first")
        .sort_values("behavior_frame_index")
        .reset_index(drop=True)
    )
    return matches

def fit_behavior_clock_from_game_frames(behavior_times, ephys_times, max_abs_residual_s=None, n_refinements=3):
    """Fit behavior_time -> ephys_time from dense game-frame sync pulses only."""
    behavior_times = np.asarray(behavior_times, dtype=float)
    ephys_times = np.asarray(ephys_times, dtype=float)
    if len(behavior_times) < 2 or len(ephys_times) < 2:
        raise ValueError("Need at least two behavior and ephys game-frame events to fit a clock.")

    behavior_interval = float(np.nanmedian(np.diff(behavior_times)))
    ephys_interval = float(np.nanmedian(np.diff(ephys_times)))
    if max_abs_residual_s is None:
        max_abs_residual_s = min(0.010, 0.45 * min(behavior_interval, ephys_interval))

    # Frame-only initial guess: align the observed frame time spans. This does not use trial starts.
    slope = (ephys_times[-1] - ephys_times[0]) / (behavior_times[-1] - behavior_times[0])
    intercept = ephys_times[0] - slope * behavior_times[0]
    stages = [{"stage": "span_seed", "slope": float(slope), "intercept": float(intercept)}]

    matches = pd.DataFrame()
    for refinement in range(n_refinements):
        matches = _nearest_unique_frame_matches(behavior_times, ephys_times, slope, intercept, max_abs_residual_s)
        if len(matches) < 2:
            raise ValueError(f"Only {len(matches)} game-frame matches survived the residual threshold.")

        slope, intercept = np.polyfit(matches["behavior_time"], matches["ephys_time"], 1)
        refit_residuals = matches["ephys_time"] - (slope * matches["behavior_time"] + intercept)
        stages.append({
            "stage": f"refine_{refinement + 1}",
            "slope": float(slope),
            "intercept": float(intercept),
            "n_matches": int(len(matches)),
            "median_abs_residual_seconds": float(np.nanmedian(np.abs(refit_residuals))),
            "max_abs_residual_seconds": float(np.nanmax(np.abs(refit_residuals))),
        })

    matches = _nearest_unique_frame_matches(behavior_times, ephys_times, slope, intercept, max_abs_residual_s)
    residuals = matches["ephys_time"] - (slope * matches["behavior_time"] + intercept)
    matches["fit_residual_seconds"] = residuals
    alignment = {
        "mode": "game_frame_clock_fit",
        "behavior_sync_event": "game_frame",
        "ephys_event": "game_frame",
        "slope": float(slope),
        "intercept": float(intercept),
        "n_matched_pulses": int(len(matches)),
        "raw_behavior_frame_count": int(len(behavior_times)),
        "raw_ephys_game_frame_count": int(len(ephys_times)),
        "max_abs_match_residual_seconds": float(max_abs_residual_s),
        "behavior_median_interval_seconds": behavior_interval,
        "ephys_median_interval_seconds": ephys_interval,
        "fit_residual_median_abs_seconds": float(np.nanmedian(np.abs(residuals))),
        "fit_residual_q95_abs_seconds": float(np.nanpercentile(np.abs(residuals), 95)),
        "fit_residual_max_abs_seconds": float(np.nanmax(np.abs(residuals))),
        "stages": stages,
    }
    return alignment, matches

alignment, frame_matches = fit_behavior_clock_from_game_frames(behavior_game_frame, ephys_frame)

print("Frame-derived behavior clock -> ephys clock alignment")
display(alignment)
display(frame_matches[["behavior_frame_index", "ephys_frame_index", "behavior_time", "ephys_time", "fit_residual_seconds"]].head())
display(pd.Series(frame_matches["fit_residual_seconds"], name="game_frame_fit_residual_seconds").describe())

In [ ]:
behavior_trial_start = sync_events.loc[sync_events["event_type"].eq("trial_start"), "t"].to_numpy(dtype=float)
ephys_trial_start = digital_events.loc[digital_events["event_name"].eq("trial_start"), "time"].to_numpy(dtype=float)
n_trial = min(len(behavior_trial_start), len(ephys_trial_start))
mapped_trial_start = alignment["slope"] * behavior_trial_start[:n_trial] + alignment["intercept"]
trial_start_residuals = ephys_trial_start[:n_trial] - mapped_trial_start

frame_residual_ms = frame_matches["fit_residual_seconds"].to_numpy() * 1000
frame_step = max(1, len(frame_matches) // 6000)

fig, axes = plt.subplots(1, 3, figsize=(16, 4), constrained_layout=True)

axes[0].plot(frame_matches["behavior_time"].to_numpy()[::frame_step], frame_residual_ms[::frame_step], ".", ms=2, alpha=0.45)
axes[0].axhline(0, color="black", lw=0.8)
axes[0].set_xlabel("behavior game-frame time (s)")
axes[0].set_ylabel("frame fit residual (ms)")
axes[0].set_title("Game-frame clock fit over time")

axes[1].hist(frame_residual_ms, bins=120, color="#334155", alpha=0.85)
axes[1].axvline(0, color="black", lw=0.8)
axes[1].set_xlabel("frame fit residual (ms)")
axes[1].set_ylabel("count")
axes[1].set_title("Matched game-frame residuals")
axes[1].set_xlim(-3, 3)
axes[2].plot(behavior_trial_start[:n_trial], trial_start_residuals * 1000, ".", ms=3)
axes[2].axhline(0, color="black", lw=0.8)
axes[2].set_xlabel("behavior trial-start time (s)")
axes[2].set_ylabel("trial-start residual (ms)")
axes[2].set_title("Trial starts validate frame-derived fit")

plt.show()

display(pd.Series(trial_start_residuals * 1000, name="trial_start_validation_residual_ms").describe())

## 4. Build the session events dataframe

This table keeps every behavior event row. Trial starts use the TTL timestamp when available. Other events use the behavior-to-ephys linear fit.

In [ ]:
def behavior_to_ephys_time(values, alignment):
    return values * alignment["slope"] + alignment["intercept"]


def safe_int(value):
    try:
        if pd.isna(value):
            return None
        return int(float(value))
    except (TypeError, ValueError):
        return None


def expand_extra_json(frame):
    extras = []
    for value in frame.get("extra", pd.Series(index=frame.index, dtype=object)):
        if isinstance(value, str) and value.strip():
            try:
                payload = json.loads(value)
            except json.JSONDecodeError:
                payload = {}
        else:
            payload = {}
        extras.append({f"extra_{key}": item for key, item in payload.items()})
    return pd.concat([frame.drop(columns=["extra"], errors="ignore"), pd.DataFrame(extras, index=frame.index)], axis=1)


def as_bool(value):
    if isinstance(value, bool):
        return value
    if pd.isna(value):
        return False
    if isinstance(value, (int, float, np.integer, np.floating)):
        return bool(value)
    return str(value).strip().lower() in {"true", "1", "yes", "y"}


def derive_trial_outcome(row):
    source_event_type = str(row.get("source_event_type", row.get("event_type", ""))).lower()
    extra_outcome = str(row.get("extra_outcome", "")).strip().lower()
    extra_reason = str(row.get("extra_reason", "")).strip().lower()

    if source_event_type == "fall_trial" or "fall" in extra_outcome or "fall" in extra_reason:
        return "fall"
    if extra_outcome in {"correct", "hit", "rewarded", "success"}:
        return "hit"
    if as_bool(row.get("rewarded", False)):
        return "hit"
    return "miss"


session_events = behavior_events.copy()
session_events.insert(0, "session_event_id", np.arange(len(session_events), dtype=int))
session_events["behavior_t_start"] = pd.to_numeric(session_events["t_start"], errors="coerce")
session_events["behavior_t_end"] = pd.to_numeric(session_events["t_end"], errors="coerce")
session_events["behavior_aligned_start_time"] = behavior_to_ephys_time(session_events["behavior_t_start"], alignment)
session_events["behavior_aligned_end_time"] = behavior_to_ephys_time(session_events["behavior_t_end"], alignment)

TRIAL_LIKE_EVENT_TYPES = ("trial", "fall_trial")
trial_like_mask = session_events["event_type"].isin(TRIAL_LIKE_EVENT_TYPES)
session_events["trial_like_ordinal"] = np.nan
session_events.loc[trial_like_mask, "trial_like_ordinal"] = np.arange(1, int(trial_like_mask.sum()) + 1, dtype=int)

trial_ttls = digital_events.loc[digital_events["event_name"].eq("trial_start")].set_index("ordinal")["time"].to_dict()
reward_ttls = digital_events.loc[digital_events["event_name"].eq("reward")].set_index("ordinal")["time"].to_dict()

session_events["ttl_trial_start_time"] = [trial_ttls.get(safe_int(value), math.nan) for value in session_events["trial_like_ordinal"]]
session_events_for_outcome = expand_extra_json(session_events)
session_event_outcomes = session_events_for_outcome.apply(derive_trial_outcome, axis=1)
session_events["reward_time"] = [
    reward_ttls.get(safe_int(reward_count), math.nan) if outcome == "hit" else math.nan
    for reward_count, outcome in zip(session_events["reward_count"], session_event_outcomes)
]

session_events["ephys_start_time"] = session_events["behavior_aligned_start_time"]
ttl_mask = session_events["ttl_trial_start_time"].notna()
session_events.loc[ttl_mask, "ephys_start_time"] = session_events.loc[ttl_mask, "ttl_trial_start_time"]
session_events["ephys_end_time"] = session_events["behavior_aligned_end_time"]
session_events["ephys_start_time_source"] = np.where(ttl_mask, "ttl_trial_start", "behavior_linear_fit")
session_events["ephys_end_time_source"] = "behavior_linear_fit"
session_events["start_alignment_residual_seconds"] = session_events["ttl_trial_start_time"] - session_events["behavior_aligned_start_time"]

session_events = expand_extra_json(session_events)

first_columns = [
    "session_event_id", "event_type", "phase_id", "trial_id", "trial_like_ordinal",
    "ephys_start_time", "ephys_end_time", "ephys_start_time_source", "ephys_end_time_source",
    "ttl_trial_start_time", "reward_time", "start_alignment_residual_seconds",
    "behavior_t_start", "behavior_t_end", "behavior_aligned_start_time", "behavior_aligned_end_time",
    "rewarded", "reward_count", "stim_id", "contrast", "orientation", "user_assisted", "x", "y", "heading",
]
ordered = [column for column in first_columns if column in session_events.columns]
ordered.extend(column for column in session_events.columns if column not in ordered and column not in {"t_start", "t_end"})
session_events = session_events[ordered].reset_index(drop=True)

print(session_events.shape)
display(session_events.head(10))
display(session_events.loc[trial_like_mask, ["session_event_id", "event_type", "trial_id", "trial_like_ordinal", "ephys_start_time", "ephys_start_time_source", "ttl_trial_start_time", "start_alignment_residual_seconds"]].head(12))
display(session_events.loc[trial_like_mask, ["session_event_id", "event_type", "trial_id", "trial_like_ordinal", "ephys_start_time", "ephys_start_time_source", "ttl_trial_start_time", "start_alignment_residual_seconds"]].tail(12))

## 5. Build frame-sync residuals and final trials_df

The dense `game_frame` sync events define the behavior-to-ephys clock transform. This residual table checks that transform in the direction we care about downstream: for every ephys-recorded game-frame TTL, find the nearest behavior/game-frame TTL after mapping behavior time into ephys time.

In [ ]:
def build_ephys_frame_sync_residuals(ephys_times, behavior_times, alignment):
    ephys_times = np.asarray(ephys_times, dtype=float)
    behavior_times = np.asarray(behavior_times, dtype=float)
    behavior_ephys_times = behavior_to_ephys_time(behavior_times, alignment)

    positions = np.searchsorted(behavior_ephys_times, ephys_times, side="left")
    rows = []
    for ephys_index, position in enumerate(positions):
        candidates = []
        if 0 <= position < len(behavior_ephys_times):
            candidates.append(position)
        if 0 <= position - 1 < len(behavior_ephys_times):
            candidates.append(position - 1)
        if not candidates:
            continue

        behavior_index = min(candidates, key=lambda idx: abs(ephys_times[ephys_index] - behavior_ephys_times[idx]))
        residual_seconds = ephys_times[ephys_index] - behavior_ephys_times[behavior_index]
        rows.append({
            "ephys_game_frame_ordinal": int(ephys_index + 1),
            "ephys_game_frame_time": float(ephys_times[ephys_index]),
            "nearest_behavior_game_frame_ordinal": int(behavior_index + 1),
            "nearest_behavior_game_frame_time": float(behavior_times[behavior_index]),
            "nearest_behavior_game_frame_ephys_time": float(behavior_ephys_times[behavior_index]),
            "residual_seconds": float(residual_seconds),
            "residual_ms": float(residual_seconds * 1000),
        })
    return pd.DataFrame(rows)


frame_sync_residuals = build_ephys_frame_sync_residuals(ephys_frame, behavior_game_frame, alignment)
display(frame_sync_residuals.head())
display(frame_sync_residuals["residual_ms"].describe())

This `trials_df` is the cleaner analysis-facing table. It keeps one row per trial-like behavior row: `trial` and `fall_trial`. The exported `event_type` is normalized to `trial`; the original label is retained in `source_event_type`. `session_start` stays in the broad `session_events` table but does not consume a trial-start TTL.

- `t_start` and `trial_start_time` are ephys trial-start TTL times.
- `reward_time` is an ephys reward TTL time when present.
- Behavior-only timing columns are mapped through the game-frame clock fit.
- `outcome` is normalized to `hit`, `miss`, or `fall`.

In [ ]:
def as_bool(value):
    if isinstance(value, bool):
        return value
    if pd.isna(value):
        return False
    if isinstance(value, (int, float, np.integer, np.floating)):
        return bool(value)
    return str(value).strip().lower() in {"true", "1", "yes", "y"}


def derive_trial_outcome(row):
    source_event_type = str(row.get("event_type", "")).lower()
    extra_outcome = str(row.get("extra_outcome", "")).strip().lower()
    extra_reason = str(row.get("extra_reason", "")).strip().lower()

    if source_event_type == "fall_trial" or "fall" in extra_outcome or "fall" in extra_reason:
        return "fall"
    if extra_outcome in {"correct", "hit", "rewarded", "success"}:
        return "hit"
    if as_bool(row.get("rewarded", False)):
        return "hit"
    return "miss"


trial_like_mask = session_events["event_type"].isin(TRIAL_LIKE_EVENT_TYPES)
trials_df = session_events.loc[trial_like_mask].copy().reset_index(drop=True)
trials_df["source_event_type"] = trials_df["event_type"]
trials_df["event_type"] = "trial"
trials_df["behavior_trial_id"] = trials_df["trial_id"]
trials_df["trial_id"] = trials_df["trial_like_ordinal"].astype("Int64")
trials_df["trial_number"] = trials_df["trial_id"]
trials_df["outcome"] = trials_df.apply(derive_trial_outcome, axis=1)

# Analysis-facing times are in the ephys timebase.
trials_df["trial_start_time"] = trials_df["ttl_trial_start_time"]
trials_df["t_start"] = trials_df["trial_start_time"]
trials_df["t_end"] = trials_df["behavior_aligned_end_time"]
trials_df.loc[trials_df["t_end"].le(trials_df["t_start"]), "t_end"] = np.nan
trials_df["reward_time"] = pd.to_numeric(trials_df["reward_time"], errors="coerce")
trials_df["rewarded"] = trials_df["reward_time"].notna() | trials_df["outcome"].eq("hit")

first_trial_columns = [
    "trial_id", "trial_number", "event_type", "outcome", "source_event_type", "behavior_trial_id",
    "t_start", "t_end", "trial_start_time", "reward_time",
    "behavior_t_start", "behavior_t_end", "behavior_aligned_start_time", "behavior_aligned_end_time",
    "start_alignment_residual_seconds", "trial_like_ordinal", "session_event_id",
    "phase_id", "rewarded", "reward_count", "stim_id", "contrast", "orientation", "user_assisted", "x", "y", "heading",
]
trial_ordered = [column for column in first_trial_columns if column in trials_df.columns]
trial_ordered.extend(column for column in trials_df.columns if column not in trial_ordered and column not in {"ephys_start_time", "ephys_end_time"})
trials_df = trials_df[trial_ordered]

display(trials_df.head(12))
display(trials_df["outcome"].value_counts(dropna=False).rename("count"))
display(trials_df[["trial_id", "source_event_type", "outcome", "t_start", "t_end", "reward_time", "start_alignment_residual_seconds"]].tail(12))

## 6. Verification checks

These are the first checks I would use before trusting the dataframe downstream.

In [ ]:
trial_like_mask = session_events["event_type"].isin(TRIAL_LIKE_EVENT_TYPES)
checks = {
    "rows": len(session_events),
    "behavior_rows": len(behavior_events),
    "trial_rows": int(session_events["event_type"].eq("trial").sum()),
    "fall_trial_rows": int(session_events["event_type"].eq("fall_trial").sum()),
    "session_start_rows": int(session_events["event_type"].eq("session_start").sum()),
    "trial_like_rows": int(trial_like_mask.sum()),
    "trials_df_rows": int(len(trials_df)),
    "trial_start_ttl_count": int(digital_events["event_name"].eq("trial_start").sum()),
    "unmatched_trial_start_ttl_count": int(digital_events["event_name"].eq("trial_start").sum() - trial_like_mask.sum()),
    "trial_like_rows_using_ttl_start": int(session_events.loc[trial_like_mask, "ephys_start_time_source"].eq("ttl_trial_start").sum()),
    "trials_df_rows_with_ttl_start": int(trials_df["trial_start_time"].notna().sum()),
    "behavior_fit_rows": int(session_events["ephys_start_time_source"].eq("behavior_linear_fit").sum()),
    "reward_ttl_count": int(digital_events["event_name"].eq("reward").sum()),
    "trials_df_reward_time_count": int(trials_df["reward_time"].notna().sum()),
    "ephys_game_frame_ttl_count": int(digital_events["event_name"].eq("game_frame").sum()),
    "frame_sync_residual_rows": int(len(frame_sync_residuals)),
}
display(pd.Series(checks, name="value"))

display(session_events["event_type"].value_counts(dropna=False).rename("count"))
display(trials_df["outcome"].value_counts(dropna=False).rename("trials_df_outcome_count"))
display(session_events["ephys_start_time_source"].value_counts(dropna=False).rename("count"))
display(session_events.loc[trial_like_mask, "start_alignment_residual_seconds"].describe())

In [ ]:
columns = ["session_event_id", "event_type", "trial_id", "trial_like_ordinal", "ephys_start_time", "ephys_start_time_source", "reward_time", "extra_outcome", "extra_reason"]
behavior_only = session_events.loc[~session_events["event_type"].isin(TRIAL_LIKE_EVENT_TYPES), [column for column in columns if column in session_events.columns]]
display(behavior_only.head(25))

large_residual_trials = session_events.loc[
    session_events["start_alignment_residual_seconds"].abs().gt(0.005),
    ["session_event_id", "trial_id", "ephys_start_time", "behavior_aligned_start_time", "start_alignment_residual_seconds"],
]
display(large_residual_trials)

## 7. Compare game/movie frame timing to ephys frame TTLs

There are multiple frame-like streams here:

- `sync_events.csv` has `game_frame` pulses from the behavior app.
- Open Ephys has NI-DAQ/PXI-6133 TTL line 6, currently treated here as the ephys-recorded version of behavior `game_frame`.
- The behavior session also saved movie frame timestamp sidecars for FLIR cameras and mirror capture.

These do not have to have identical counts. This section compares counts, rates, nearest-neighbor timing residuals, and drift.

In [ ]:
def summarize_time_series(name, times):
    times = np.asarray(times, dtype=float)
    diffs = np.diff(times)
    return {
        "name": name,
        "count": len(times),
        "first_s": float(times[0]) if len(times) else np.nan,
        "last_s": float(times[-1]) if len(times) else np.nan,
        "span_s": float(times[-1] - times[0]) if len(times) > 1 else np.nan,
        "mean_dt_s": float(np.nanmean(diffs)) if len(diffs) else np.nan,
        "median_dt_s": float(np.nanmedian(diffs)) if len(diffs) else np.nan,
        "p99_dt_s": float(np.nanpercentile(diffs, 99)) if len(diffs) else np.nan,
        "max_dt_s": float(np.nanmax(diffs)) if len(diffs) else np.nan,
        "effective_hz": float((len(times) - 1) / (times[-1] - times[0])) if len(times) > 1 and times[-1] > times[0] else np.nan,
    }


def nearest_residuals(reference_times, query_times):
    reference_times = np.asarray(reference_times, dtype=float)
    query_times = np.asarray(query_times, dtype=float)
    positions = np.searchsorted(reference_times, query_times)
    positions = np.clip(positions, 1, len(reference_times) - 1)
    left = reference_times[positions - 1]
    right = reference_times[positions]
    nearest = np.where(np.abs(query_times - left) <= np.abs(query_times - right), left, right)
    return nearest - query_times


ephys_frame_times = digital_events.loc[digital_events["event_name"].eq("game_frame"), "time"].to_numpy(dtype=float)
behavior_game_frame_times = sync_events.loc[sync_events["event_type"].eq("game_frame"), "t"].to_numpy(dtype=float)
behavior_game_frame_ephys = behavior_to_ephys_time(behavior_game_frame_times, alignment)

series_summary = [
    summarize_time_series("ephys-recorded game_frame line 6", ephys_frame_times),
    summarize_time_series("behavior sync_events game_frame", behavior_game_frame_times),
    summarize_time_series("behavior game_frame mapped to ephys", behavior_game_frame_ephys),
]
display(pd.DataFrame(series_summary))

game_frame_residuals = nearest_residuals(ephys_frame_times, behavior_game_frame_ephys)
display(pd.Series(game_frame_residuals * 1000, name="nearest_ephys_minus_behavior_game_frame_ms").describe())

In [ ]:
movie_timestamp_files = {
    "flir_cam0": BEHAVIOR_FOLDER / "camera" / "flir_2026-06-29_14-29-26_cam0_22588505_frame_times_s.npy",
    "flir_cam1": BEHAVIOR_FOLDER / "camera" / "flir_2026-06-29_14-29-26_cam1_20173661_frame_times_s.npy",
    "mirror": BEHAVIOR_FOLDER / "mirror_capture" / "mirror_capture_frame_times_s.npy",
}

movie_rows = []
movie_residuals = {}
for name, path in movie_timestamp_files.items():
    if not path.exists():
        print(f"missing {name}: {path}")
        continue
    behavior_movie_times = np.load(path)
    ephys_movie_times = behavior_to_ephys_time(behavior_movie_times, alignment)
    residual_ms = nearest_residuals(ephys_frame_times, ephys_movie_times) * 1000
    movie_residuals[name] = residual_ms
    row = summarize_time_series(name, behavior_movie_times)
    row.update({
        "mapped_first_s": float(ephys_movie_times[0]) if len(ephys_movie_times) else np.nan,
        "mapped_last_s": float(ephys_movie_times[-1]) if len(ephys_movie_times) else np.nan,
        "nearest_ephys_residual_median_ms": float(np.nanmedian(residual_ms)) if len(residual_ms) else np.nan,
        "nearest_ephys_residual_mad_ms": float(np.nanmedian(np.abs(residual_ms - np.nanmedian(residual_ms)))) if len(residual_ms) else np.nan,
        "nearest_ephys_abs_residual_p95_ms": float(np.nanpercentile(np.abs(residual_ms), 95)) if len(residual_ms) else np.nan,
        "nearest_ephys_abs_residual_max_ms": float(np.nanmax(np.abs(residual_ms))) if len(residual_ms) else np.nan,
    })
    movie_rows.append(row)

movie_comparison = pd.DataFrame(movie_rows)
display(movie_comparison)

### Causal frame matching test

If behavior `game_frame` sync pulses cause ephys frame TTLs, then a behavior frame should precede its matching ephys edge. Because counts differ, this does not use same-index matching. It asks two directional questions:

- For each ephys frame TTL, how long since the previous behavior `game_frame`?
- For each behavior `game_frame`, how long until the next ephys frame TTL?

A tight, near-constant positive lag would support using game frames as a high-density clock alignment signal. A broad or multi-modal lag argues that the streams are related but not one-to-one.

In [ ]:
def previous_event_lag(reference_times, query_times):
    """Return query - previous reference for every query with an earlier reference."""
    reference_times = np.asarray(reference_times, dtype=float)
    query_times = np.asarray(query_times, dtype=float)
    positions = np.searchsorted(reference_times, query_times, side="right") - 1
    valid = positions >= 0
    return query_times[valid] - reference_times[positions[valid]], valid


def next_event_lag(reference_times, query_times):
    """Return next reference - query for every query with a later reference."""
    reference_times = np.asarray(reference_times, dtype=float)
    query_times = np.asarray(query_times, dtype=float)
    positions = np.searchsorted(reference_times, query_times, side="left")
    valid = positions < len(reference_times)
    return reference_times[positions[valid]] - query_times[valid], valid


ephys_after_previous_behavior_lag, _ = previous_event_lag(behavior_game_frame_ephys, ephys_frame_times)
next_ephys_after_behavior_lag, _ = next_event_lag(ephys_frame_times, behavior_game_frame_ephys)
next_behavior_after_ephys_lag, _ = next_event_lag(behavior_game_frame_ephys, ephys_frame_times)

lag_rows = []
for name, lags in [
    ("ephys - previous behavior game_frame", ephys_after_previous_behavior_lag),
    ("next ephys - behavior game_frame", next_ephys_after_behavior_lag),
    ("next behavior game_frame - ephys", next_behavior_after_ephys_lag),
]:
    lag_ms = lags * 1000
    lag_rows.append({
        "comparison": name,
        "count": len(lag_ms),
        "median_ms": float(np.nanmedian(lag_ms)),
        "p01_ms": float(np.nanpercentile(lag_ms, 1)),
        "p05_ms": float(np.nanpercentile(lag_ms, 5)),
        "p95_ms": float(np.nanpercentile(lag_ms, 95)),
        "p99_ms": float(np.nanpercentile(lag_ms, 99)),
        "max_ms": float(np.nanmax(lag_ms)),
        "within_1ms_fraction": float(np.nanmean(lag_ms <= 1)),
        "within_5ms_fraction": float(np.nanmean(lag_ms <= 5)),
        "within_17ms_fraction": float(np.nanmean(lag_ms <= 17)),
    })

lag_summary = pd.DataFrame(lag_rows)
display(lag_summary)


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4), constrained_layout=True)
bins = np.linspace(0, 35, 160)
ax.hist(ephys_after_previous_behavior_lag * 1000, bins=bins, alpha=0.45, label="ephys - previous behavior")
ax.hist(next_ephys_after_behavior_lag * 1000, bins=bins, alpha=0.45, label="next ephys - behavior")
ax.hist(next_behavior_after_ephys_lag * 1000, bins=bins, alpha=0.45, label="next behavior - ephys")
ax.set_xlabel("lag (ms)")
ax.set_ylabel("count")
ax.set_title("Directional frame matching lag distributions")
ax.legend(frameon=False)
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7), constrained_layout=True)

axes[0].hist(game_frame_residuals * 1000, bins=120, color="#334155", alpha=0.85)
axes[0].set_title("Behavior sync_events game_frame -> nearest ephys frame TTL")
axes[0].set_xlabel("nearest ephys TTL minus mapped behavior frame time (ms)")
axes[0].set_ylabel("count")

for name, residual_ms in movie_residuals.items():
    axes[1].hist(residual_ms, bins=120, alpha=0.45, label=name)
axes[1].set_title("Movie timestamp sidecars -> nearest ephys frame TTL")
axes[1].set_xlabel("nearest ephys TTL minus mapped movie frame time (ms)")
axes[1].set_ylabel("count")
axes[1].legend(frameon=False)

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4), constrained_layout=True)

sample_step = max(1, len(game_frame_residuals) // 5000)
ax.plot(behavior_game_frame_ephys[::sample_step], (game_frame_residuals * 1000)[::sample_step], ".", ms=2, alpha=0.5, label="sync_events game_frame")

for name, path in movie_timestamp_files.items():
    if name not in movie_residuals:
        continue
    times = behavior_to_ephys_time(np.load(path), alignment)
    residual_ms = movie_residuals[name]
    step = max(1, len(residual_ms) // 3000)
    ax.plot(times[::step], residual_ms[::step], ".", ms=2, alpha=0.5, label=name)

ax.axhline(0, color="black", lw=0.8)
ax.set_xlabel("ephys time (s)")
ax.set_ylabel("nearest ephys frame residual (ms)")
ax.set_title("Frame timing residuals over session")
ax.legend(frameon=False, ncol=2)
plt.show()

## 8. Compare against the pipeline-generated table

If `pipeline.mouse_arena_session` has been run, this checks whether this notebook reconstruction matches the saved CSV for key timing columns.

In [ ]:
saved_csv = OUTPUT_FOLDER / "session_events.csv"
if saved_csv.exists():
    saved = pd.read_csv(saved_csv)
    key_columns = [
        "session_event_id", "event_type", "trial_id", "ephys_start_time", "ephys_end_time",
        "ttl_trial_start_time", "reward_time", "behavior_aligned_start_time", "start_alignment_residual_seconds",
    ]
    common = [column for column in key_columns if column in saved.columns and column in session_events.columns]
    merged = saved[common].merge(session_events[common], on="session_event_id", suffixes=("_saved", "_notebook"))
    diff_columns = []
    for column in common:
        if column == "session_event_id":
            continue
        left = f"{column}_saved"
        right = f"{column}_notebook"
        if left in merged.columns and right in merged.columns and pd.api.types.is_numeric_dtype(merged[left]):
            merged[f"{column}_abs_diff"] = (merged[left] - merged[right]).abs()
            diff_columns.append(f"{column}_abs_diff")
    display(merged[diff_columns].max().rename("max_abs_diff"))
    display(merged.head())
else:
    print(f"No saved table found at {saved_csv}")

In [ ]:
events.head()

## 9. Save notebook-generated tables

This writes notebook-generated sandbox outputs into the recording-local `nwb/` folder. These are still inspection products, not official pipeline outputs yet.

In [ ]:
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

trials_csv = OUTPUT_FOLDER / "trials_df_from_notebook.csv"
frame_sync_csv = OUTPUT_FOLDER / "frame_sync_residuals_from_notebook.csv"
session_events_csv = OUTPUT_FOLDER / "session_events_from_notebook.csv"

trials_df.to_csv(trials_csv, index=False)
frame_sync_residuals.to_csv(frame_sync_csv, index=False)
session_events.to_csv(session_events_csv, index=False)

print(trials_csv)
print(frame_sync_csv)
print(session_events_csv)